This notebook is for building our own lightweight version of Claimify. Claimify uses a 4-stage pipeline:
1. Sentence Splitting: Breaks text into individual sentences with surrounding context
2. Selection: Filters for sentences containing verifiable propositions, excluding opinions and speculation
3. Disambiguation: Resolves ambiguities or discards sentences that cannot be clarified
4. Decomposition: Breaks down sentences into atomic, self-contained factual claims

In [1]:
import re
import nltk
import spacy
import torch
from fastcoref.modeling import FCoref, FCorefModel
from transformers import pipeline

In [2]:
# Download the sentence tokenizer for sentence splitting
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
# Initialize the fastcoref model for disambiguation
if not hasattr(FCorefModel, "all_tied_weights_keys"):
    FCorefModel.all_tied_weights_keys = {}

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
coref_model = FCoref(device=device)

04/07/2026 13:13:30 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/07/2026 13:13:30 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/07/2026 13:13:40 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04/07/2026 13:13:40 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/tokenizer_config.json "HTTP/1.1 200 OK"
04/07/2026 13:13:40 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04/07/2026 13:13:40 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/tree/main?recursive=true&e

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

FCorefModel LOAD REPORT from: biu-nlp/f-coref
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
04/07/2026 13:13:51 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref "HTTP/1.1 200 OK"
04/07/2026 13:13:51 - INFO - 	 missing_keys: []
04/07/2026 13:13:51 - INFO - 	 unexpected_keys: ['roberta.embeddings.position_ids']
04/07/2026 13:13:51 - INFO - 	 mismatched_keys: []
04/07/2026 13:13:51 - INFO - 	 error_msgs: []
04/07/2026 13:13:51 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [4]:
# Initialize a lightweight zero-shot classification model for selection
claim_classifier = pipeline(
    "zero-shot-classification", model="valhalla/distilbart-mnli-12-3", device=-1
)

04/07/2026 13:13:51 - INFO - 	 HTTP Request: HEAD https://huggingface.co/valhalla/distilbart-mnli-12-3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/07/2026 13:13:51 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/valhalla/distilbart-mnli-12-3/ef9a58ce6a9cd44cd0d4c2f7db1cd67f81019a8b/config.json "HTTP/1.1 200 OK"
04/07/2026 13:13:51 - INFO - 	 HTTP Request: HEAD https://huggingface.co/valhalla/distilbart-mnli-12-3/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
04/07/2026 13:13:51 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3 "HTTP/1.1 200 OK"
04/07/2026 13:13:51 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/commits/main "HTTP/1.1 200 OK"
04/07/2026 13:13:51 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/discussions?p=0 "HTTP/1.1 200 OK"
04/07/2026 13:13:51 - INFO - 	 HTTP Request: GET https://h

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

04/07/2026 13:13:52 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04/07/2026 13:13:52 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


In [5]:
# Load spacy to use part of speech for decomposition
nlp = spacy.load("en_core_web_sm")

In [6]:
def resolve_coreferences(text):
    """
    Replaces pronouns and implicit references in the text with their explicit entities.
    """
    # Predict coreference clusters
    preds = coref_model.predict(texts=[text])

    # Get clusters as character start/end indices
    clusters = preds[0].get_clusters(as_strings=False)

    replacements = []
    for cluster in clusters:
        # The first mention in a cluster is usually the explicit entity (the antecedent)
        primary_start, primary_end = cluster[0]
        primary_text = text[primary_start:primary_end]

        # Replace all subsequent mentions (usually pronouns) with the primary text
        for mention_start, mention_end in cluster[1:]:
            replacements.append((mention_start, mention_end, primary_text))

    # Sort replacements in reverse order of their start index
    replacements.sort(key=lambda x: x[0], reverse=True)

    # Apply replacements
    resolved_text = text
    for start, end, rep_text in replacements:
        resolved_text = resolved_text[:start] + rep_text + resolved_text[end:]

    return resolved_text

In [7]:
def format_claim(text):
    """Helper function to clean up capitalization and punctuation."""
    # 1. Strip trailing whitespace and dangling commas
    text = re.sub(r"[,\s]+$", "", text.strip())

    # 2. Fix spaces before periods or commas (e.g., "year ." -> "year.")
    text = re.sub(r"\s+([.,])", r"\1", text)

    # 3. Ensure it ends with a period
    if not text.endswith("."):
        text += "."

    # 4. Capitalize the first letter (without changing the rest of the string)
    if len(text) > 0:
        text = text[0].upper() + text[1:]

    return text

In [8]:
def decompose_sentence(sentence):
    doc = nlp(sentence)

    # 1. Find the ROOT verb of the sentence
    root = next((t for t in doc if t.dep_ == "ROOT"), None)
    if not root:
        return [format_claim(sentence)]

    # Check if there are any conjunct verbs attached to the root
    conjunct_verbs = [t for t in root.children if t.dep_ == "conj" and t.pos_ == "VERB"]

    # If there are no compound predicates, return the sentence as is
    if not conjunct_verbs:
        return [format_claim(sentence)]

    claims = []

    # 3.Identify the main subject AND any shared auxiliary verbs (like "will", "has", "is")
    subjects = [t for t in root.children if "subj" in t.dep_]
    if not subjects:
        return [format_claim(sentence)]

    subject_phrase = "".join([t.text_with_ws for t in subjects[0].subtree]).strip()
    auxs = [t for t in root.children if t.dep_ == "aux"]
    aux_phrase = "".join([t.text_with_ws for t in auxs]).strip()

    # This is the prefix we will attach to the second verb (e.g., "The infrastructure bill will")
    prefix = f"{subject_phrase} {aux_phrase}".strip()

    # 4. Create Claim 1: The original sentence minus the second verb and the "and"
    ignore_tokens = set()
    for conj in conjunct_verbs:
        ignore_tokens.update(
            [t.i for t in conj.subtree]
        )  # Ignore the second verb's phrase
    ignore_tokens.update(
        [t.i for t in root.children if t.dep_ == "cc"]
    )  # Ignore the conjunction

    # Reassemble using the exact original token order
    claim1_tokens = [t for t in doc if t.i not in ignore_tokens]
    claim1 = "".join([t.text_with_ws for t in claim1_tokens]).strip()
    claims.append(format_claim(claim1))

    # 5. Create Claim 2: Apply the prefix to the conjunct verb phrase
    for conj in conjunct_verbs:
        # Sort tokens to maintain correct word order
        conj_tokens = sorted(list(conj.subtree), key=lambda x: x.i)
        conj_phrase = "".join([t.text_with_ws for t in conj_tokens]).strip()

        atomic_claim = f"{prefix} {conj_phrase}"
        claims.append(format_claim(atomic_claim))

    return claims

In [9]:
def lightweight_claimify(text, threshold=0.6):
    """
    The complete, keyless 4-stage claim extraction pipeline.
    """
    # Stage 1: Disambiguation (Coreference Resolution)—we're doing first instead of third because it works best with our lightweight, API-free approach
    disambiguated_text = resolve_coreferences(text)

    # Stage 2: Sentence Splitting
    complex_sentences = nltk.sent_tokenize(disambiguated_text)
    atomic_sentences = []

    # Stage 3: Decomposition
    for sent in complex_sentences:
        # Only try to decompose longer sentences with conjunctions
        if " and " in sent.lower() or " but " in sent.lower() or "," in sent:
            decomposed = decompose_sentence(sent)
            atomic_sentences.extend(decomposed)
        else:
            atomic_sentences.append(sent)

    extracted_claims = []

    # Stage 4: Selection (Filtering opinions vs. facts)
    for claim in atomic_sentences:
        if len(claim.split()) < 4:
            continue

        result = claim_classifier(
            claim,
            candidate_labels=["factual claim", "personal opinion"],
            multi_label=False,
        )

        # Keep only the confident factual claims
        if result["labels"][0] == "factual claim" and result["scores"][0] >= threshold:
            extracted_claims.append(claim)

    # Clean up duplicates that can happen during decomposition
    return list(set(extracted_claims))

In [17]:
def print_claims(sample_article):
    """
    Function prints out extracted claims from sample text for easy and readable testing.
    """
    claims = lightweight_claimify(sample_article, threshold=0.5)
    if len(claims) != 0:
        print("\n--- Extracted Claims ---")
        for claim in claims:
            print(claim)
    else:
        print("\n--- No Claims Extracted ---")
    return claims

First, let's check we're able to handle all the key challenges that Claimify was able to handle: opinions, unresolved ambiguity, attribution, and complex sentences.

In [18]:
# Example with opinions, unresolved ambiguity, attribution, and complex sentences
print("\n=== TEST: COMPREHENSIVE INTEGRATION ===")
complex_text = """
I think the new policies in the U.S. are an absolute disaster.
The inflation rate in the country rose by 4.2% last quarter.
It is the worst economic decision in history!
According to the report, the city council voted 5-2 to pass the infrastructure bill.
This bill will go into effect next year and raise taxes 5%.
"""

extracted_claims = print_claims(complex_text)

# Check for the expected factual claims
has_inflation = any(
    "inflation rate" in c.lower() and "4.2%" in c for c in extracted_claims
)
has_council = any(
    "city council voted 5-2" in c.lower() and "according to the report" in c.lower()
    for c in extracted_claims
)
has_effect = any("go into effect next year" in c.lower() for c in extracted_claims)
has_taxes = any("raise taxes 5%" in c.lower() for c in extracted_claims)

# Check that opinions were successfully excluded
has_disaster = any("disaster" in c.lower() for c in extracted_claims)
has_worst = any("worst" in c.lower() for c in extracted_claims)

if (
    len(extracted_claims) == 4
    and has_inflation
    and has_council
    and has_effect
    and has_taxes
    and not has_disaster
    and not has_worst
):
    print(
        "✅ PASS: Successfully handled opinions, coreference resolution, attribution, and atomicity in a single complex block."
    )
else:
    print("❌ FAIL: The model failed to perfectly extract the 4 target claims.")

    # Provide specific debugging feedback based on what went wrong
    if has_disaster or has_worst:
        print(
            "  -> Failure Point: Did not filter out the subjective opinions ('disaster' / 'worst')."
        )
    if not (has_effect and has_taxes):
        print(
            "  -> Failure Point: Failed to decompose the compound verb ('will go into effect... and raise taxes')."
        )
    if not has_council:
        print(
            "  -> Failure Point: Missed the city council vote or dropped the 'According to the report' attribution."
        )
    if not has_inflation:
        print("  -> Failure Point: Missed the inflation statistic.")
    if len(extracted_claims) != 4 and not (has_disaster or has_worst):
        print(
            f"  -> Failure Point: Expected 4 claims, but extracted {len(extracted_claims)}."
        )

04/07/2026 13:14:20 - INFO - 	 Tokenize 1 inputs...



=== TEST: COMPREHENSIVE INTEGRATION ===


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/07/2026 13:14:20 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


--- Extracted Claims ---
The infrastructure bill will go into effect next year.
The inflation rate in the U.S. rose by 4.2% last quarter.
The infrastructure bill will raise taxes 5%.
According to the report, the city council voted 5-2 to pass the infrastructure bill.
✅ PASS: Successfully handled opinions, coreference resolution, attribution, and atomicity in a single complex block.


Now, let's check the super challenging examples from the Claimify paper. This is where our lightweight model is not as good with some difficult or unusual examples.

In [19]:
# Example of complex opinion from Claimify paper, result should be no claims
claims_1 = print_claims(
    """Addressing emerging market challenges will require comprehensive strategies that consider economic stability, food security, and public health."""
)
if len(claims_1) == 0:
    print("✅ PASS: Correctly extracted 0 claims from a normative/opinion statement.")
else:
    print(
        "❌ FAIL: Extracted claims from an opinion. Standard models usually fail this by treating recommendations as facts."
    )

04/07/2026 13:14:24 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/07/2026 13:14:24 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


--- No Claims Extracted ---
✅ PASS: Correctly extracted 0 claims from a normative/opinion statement.


In [20]:
# Example of claim with fluff from Claimify paper, result should be just that there is a partnership between John and Jane, nothing else. Worst case result would be thinking the whole sentence is a claim.
claims_2 = print_claims(
    """The partnership between John and Jane illustrates the importance of collaboration."""
)
if len(claims_2) == 1 and "illustrates" not in claims_2[0].lower():
    print(
        "✅ PASS: Successfully isolated the verifiable fact (the partnership) and stripped the subjective fluff."
    )
elif len(claims_2) == 0:
    print(
        "❌ PARTIAL FAIL: Extracted no claims. The model likely discarded the whole sentence because of the subjective verb."
    )
else:
    print("❌ FAIL: Included subjective fluff in the claim.")

04/07/2026 13:14:26 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/07/2026 13:14:26 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


--- Extracted Claims ---
The partnership between John and Jane illustrates the importance of collaboration.
❌ FAIL: Included subjective fluff in the claim (or extracted multiple incorrect claims).


In [21]:
# Example of unresolved ambiguity from Claimify paper, result should be no claims technically
claims_3 = print_claims(
    """Countries like Afghanistan and Sudan have experienced similar challenges to those of Libya."""
)
if len(claims_3) == 0:
    print(
        "✅ PASS: Correctly rejected the sentence because 'those' cannot be resolved without context."
    )
else:
    print(
        "❌ PARTIAL FAIL: Extracted a claim but with unresolved context. A human fact-checker wouldn't know what 'those' refers to or which challenges in particular."
    )

04/07/2026 13:14:44 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/07/2026 13:14:44 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


--- Extracted Claims ---
Countries like Afghanistan and Sudan have experienced similar challenges to those of Libya.
❌ PARTIAL FAIL: Extracted a claim but with unresolved context. A human fact-checker wouldn't know what 'those' refers to or which challenges in particular.


In [22]:
# Example of important attribution from Claimify paper, result should include that the U.N. said this and attribute it to them
claims_4 = print_claims("""The U.N. said the contaminated water caused illness.""")
if any("u.n." in c.lower() or "un " in c.lower() for c in claims_4):
    print("✅ PASS: Successfully preserved the attribution to the U.N.")
elif len(claims_4) > 0:
    print(
        "❌ FAIL: Stripped the attribution. (e.g., Outputting 'Contaminated water caused illness' states it as an absolute truth rather than a quoted statement)."
    )
else:
    print("❌ FAIL: Failed to extract any claims.")

04/07/2026 13:14:51 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/07/2026 13:14:51 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


--- Extracted Claims ---
The U.N. said the contaminated water caused illness.
✅ PASS: Successfully preserved the attribution to the U.N.


In [23]:
# Example of complex multi-subject atomicity from Claimify paper, result should say that California implemented a plastic bag ban and separately that New York did
claims_5 = print_claims("""California and New York implemented a plastic bag ban.""")
has_california_alone = any(
    "california" in c.lower() and "new york" not in c.lower() for c in claims_5
)
has_new_york_alone = any(
    "new york" in c.lower() and "california" not in c.lower() for c in claims_5
)

if len(claims_5) >= 2 and has_california_alone and has_new_york_alone:
    print(
        "✅ PASS: Successfully decomposed the compound subject into independent atomic claims."
    )
elif len(claims_5) == 1:
    print("❌ PARTIAL FAIL: Left the compound subjects together as a single claim.")
else:
    print("❌ FAIL: Failed to extract claims or broke the sentence incorrectly.")

04/07/2026 13:14:54 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/07/2026 13:14:54 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


--- Extracted Claims ---
California and New York implemented a plastic bag ban.
❌ PARTIAL FAIL: Left the compound subjects together as a single claim.
